# Build updated metadata
One row per h5 file in the `PCD_Spatial` project store, joined to the cohort metadata (Oncology sheet). Files whose TIS ID is missing from the cohort sheet, or whose tissue is non-PCD, are flagged. Writes `../reference/updated_metadata.csv`.

In [ ]:
import os
import re
from pathlib import Path

import hisepy as hp
import pandas as pd

REFERENCE_DIR = Path('../reference')
H5_SUFFIX = '_sample_filtered_feature_bc_matrix.h5'
EXPECTED_UNFLAGGED_FILES = 68
NON_PCD_TISSUES = {'tonsil', 'lymph node'}

## File list from the project store

In [ ]:
store = hp.list_files_in_project_store('PCD_Spatial')
store['basename'] = store['name'].fillna('').map(os.path.basename)
h5_store = store[store['basename'].str.endswith(H5_SUFFIX)]

duplicated = h5_store.loc[h5_store['basename'].duplicated(keep=False), 'name'].tolist()
assert not duplicated, f'Duplicate basenames in the store: {duplicated}'

## Parse sample identifiers from the filenames

In [ ]:
def extract_tis(fname):
    match = re.search(r'(TIS\d{5}(?:-\d{3}(?:-\d{3})?)?)', fname)
    return match.group(1) if match else 'unknown'


def infer_project(fname):
    upper = fname.upper()
    if 'EXP00998' in upper or 'EXP998' in upper:
        return 'aacr1'
    if 'EXP01109' in upper and 'MULTIPLEX' in upper:
        return 'aacr2-mp'
    if 'EXP01109' in upper:
        return 'aacr2-sp'
    if 'EXP01349' in upper and 'MULTIPLEX' in upper:
        return 'aacr3-mp'
    if 'EXP01349' in upper:
        return 'aacr3-sp'
    if 'EXP01387' in upper:
        return 'aacr4'
    return 'unknown'


def tis_base(tis_id):
    """TIS05684-001-004 -> TIS05684-001; bare TIS05684 -> TIS05684-001 (Multiplex names omit the suffix)."""
    match = re.match(r'(TIS\d{5}-\d{3})', tis_id)
    if match:
        return match.group(1)
    bare = re.match(r'TIS\d{5}', tis_id)
    return bare.group() + '-001' if bare else tis_id


files = pd.DataFrame({
    'file_path': h5_store['name'].values,
    'source_file': h5_store['basename'].str.replace(H5_SUFFIX, '', regex=False).values,
    'TIS_ID': h5_store['basename'].map(extract_tis).values,
    'project': h5_store['basename'].map(infer_project).values,
})
files['TIS_ID_base'] = files['TIS_ID'].map(tis_base)

## Join cohort metadata

In [ ]:
cohort = pd.read_excel(REFERENCE_DIR / 'cohort_metadata_all.xlsx', sheet_name='Oncology')
cohort_columns = {
    'AIFI Barcode': 'TIS_ID_base',
    'Subject ID': 'subject_id',
    'For Experiments': 'cohort_label',
    'Tissue Type': 'tissue_type',
    'Sample Site': 'sample_site',
    'Disease State': 'disease_state',
    'Patient Age at collection': 'age_at_collection',
    'Sex': 'sex',
    'Race/Ethnicity': 'race_ethnicity',
    'Treatment (if known)': 'treatment',
    '%Tumor': 'pct_tumor',
    'Collection Year': 'collection_year',
    'Biopsy or Resection': 'biopsy_or_resection',
}
cohort = cohort[list(cohort_columns)].dropna(subset=['AIFI Barcode']).rename(columns=cohort_columns)

merged = files.merge(cohort, on='TIS_ID_base', how='left')

## Flag files to exclude

In [ ]:
def make_flag(row):
    if pd.isna(row['tissue_type']) and pd.isna(row['disease_state']) and pd.isna(row['subject_id']):
        return 'NOT_IN_COHORT_METADATA'
    tissue = '' if pd.isna(row['tissue_type']) else str(row['tissue_type']).strip().lower()
    if tissue in NON_PCD_TISSUES:
        return f'NON_PCD_TISSUE:{row["tissue_type"]}'
    disease = '' if pd.isna(row['disease_state']) else str(row['disease_state']).strip().lower()
    if disease in ('normal', 'nan', '') and 'normal' in str(row['cohort_label']).lower():
        return 'NORMAL_CONTROL'
    return ''


merged['flag'] = merged.apply(make_flag, axis=1)
merged['in_cohort_metadata'] = merged['flag'] != 'NOT_IN_COHORT_METADATA'

column_order = [
    'file_path', 'source_file', 'TIS_ID', 'TIS_ID_base', 'project',
    'in_cohort_metadata', 'flag',
    'subject_id', 'cohort_label', 'tissue_type', 'sample_site',
    'disease_state', 'age_at_collection', 'sex', 'race_ethnicity',
    'treatment', 'pct_tumor', 'collection_year', 'biopsy_or_resection',
]
merged = merged[column_order]

## Checks, then write
The unflagged files are the ones downstream notebooks download and load. Nothing is written if their count or identity differs from what is expected or from the existing csv.

In [ ]:
unflagged = set(merged.loc[merged['flag'] == '', 'source_file'])
assert len(unflagged) == EXPECTED_UNFLAGGED_FILES, f'{len(unflagged)} unflagged files, expected {EXPECTED_UNFLAGGED_FILES}'

output_path = REFERENCE_DIR / 'updated_metadata.csv'
if output_path.exists():
    previous = pd.read_csv(output_path)
    previous_unflagged = set(previous.loc[previous['flag'].isna(), 'source_file'])
    assert unflagged == previous_unflagged, (
        f'only in store: {sorted(unflagged - previous_unflagged)}; '
        f'only in existing csv: {sorted(previous_unflagged - unflagged)}'
    )

merged.to_csv(output_path, index=False)
merged['flag'].value_counts()